In [ ]:
import os
os.environ["DISABLE_WIDGETS"] = "true"

In [ ]:
from typing import List, TypedDict

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
docs = (
    PyPDFLoader("documents\Corporate_Policy_Compendium_Vol1.pdf").load() +
    PyPDFLoader("documents\Corporate_Policy_Compendium_Vol2.pdf").load()
)

In [ ]:
len(docs)

In [ ]:
docs[5].metadata

In [ ]:
print(docs[5].page_content)

In [ ]:
import re

def is_main_section(line):
    # e.g., "4. PERFORMANCE MANAGEMENT SYSTEM"
    return re.match(r"^\d+\.\s+[A-Z][A-Z\s&]+$", line)

def is_subsection(line):
    # e.g., "3.8 Security and Confidentiality"
    return re.match(r"^\d+\.\d+\s+[A-Z][a-zA-Z\s]+$", line)

def parse_document(text):
    blocks = []

    current_section = None
    current_subsection = None
    buffer = []

    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        if not line:
            continue

        #  Ignore bullet points (keep as content)
        if line.startswith("•"):
            buffer.append(line)
            continue

        #  MAIN SECTION DETECTION
        if is_main_section(line):

            # save previous block
            if buffer:
                blocks.append({
                    "section": current_section,
                    "subsection": current_subsection,
                    "content": "\n".join(buffer)
                })
                buffer = []

            current_section = line.split(".")[0]
            current_subsection = None

        #  SUBSECTION DETECTION
        elif is_subsection(line):

            # save previous block
            if buffer:
                blocks.append({
                    "section": current_section,
                    "subsection": current_subsection,
                    "content": "\n".join(buffer)
                })
                buffer = []

            current_subsection = line.split()[0]

            #  Infer section if missing
            if current_section is None:
                current_section = current_subsection.split(".")[0]

        #  NUMBERED LIST HANDLING (CRITICAL FIX)
        elif re.match(r"^\d+\.\s+", line):

            #  If inside subsection → treat as content
            if current_subsection is not None:
                buffer.append(line)
                continue

            #  If not inside subsection, ignore as structure
            buffer.append(line)
            continue

        # NORMAL TEXT
        buffer.append(line)

    # save last block
    if buffer:
        blocks.append({
            "section": current_section,
            "subsection": current_subsection,
            "content": "\n".join(buffer)
        })

    return blocks

In [ ]:
new_docs = []

for doc in docs:
    blocks = parse_document(doc.page_content)

    for block in blocks:
        new_docs.append(
            Document(
                page_content=block["content"],   # ✅ STRING
                metadata={
                    "section": block["section"],
                    "subsection": block["subsection"],
                    **doc.metadata   # keep original metadata like page
                }
            )
        )

In [ ]:
len(new_docs)

In [ ]:
print(new_docs[5])

In [ ]:
def smart_chunk(documents, splitter, max_len=800):
    final_docs = []

    for doc_idx, doc in enumerate(documents):

        if len(doc.page_content) > max_len:
            chunks = splitter.split_documents([doc])

            for i, chunk in enumerate(chunks):
                chunk.metadata["chunk_id"] = f"{doc_idx}_{i}"
                final_docs.append(chunk)

        else:
            doc.metadata["chunk_id"] = f"{doc_idx}_0"
            final_docs.append(doc)

    return final_docs

In [ ]:
# 2) Chunk
splitter=RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = smart_chunk(new_docs,splitter)
# 3) Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:
len(chunks)

In [ ]:
print(chunks[8])
print("                                %%%%%%%%%%%%%%%%")
print(chunks[10])

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
def filter_documents(docs, section=None, subsection=None):
    filtered = []

    for doc in docs:
        if section and doc.metadata.get("section") != section:
            continue

        if subsection and doc.metadata.get("subsection") != subsection:
            continue

        filtered.append(doc)

    return filtered

In [ ]:
def extract_query_filters(query):
    import re

    section = None
    subsection = None

    sec_match = re.search(r"section\s+(\d+)", query.lower())
    sub_match = re.search(r"(\d+\.\d+)", query)

    if sec_match:
        section = sec_match.group(1)

    if sub_match:
        subsection = sub_match.group(1)

    return section, subsection

In [ ]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
    citations:List[dict]

In [ ]:
# query rewritting for better retrieval

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")


In [ ]:
def rerank_documents(query, docs):
    pairs = [(query, doc.page_content) for doc in docs]

    scores = reranker.predict(pairs)

    # attach scores
    doc_scores = list(zip(docs, scores))

    # sort by score (descending)
    doc_scores.sort(key=lambda x: x[1], reverse=True)

    # return top 5 docs
    return doc_scores[:10]

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 10

In [ ]:
def deduplicate_docs(docs):
    seen = set()
    unique_docs = []

    for doc in docs:
        content = doc.page_content.strip()

        if content not in seen:
            seen.add(content)
            unique_docs.append(doc)

    return unique_docs

In [ ]:
def extract_citations(docs):
    citations = []

    for doc in docs:
        meta = doc.metadata

        citations.append({
            "chunk_id": meta.get("chunk_id"),
            "section": meta.get("section"),
            "subsection": meta.get("subsection"),
            "source": meta.get("source"),
            "page": meta.get("page")
        })

    return citations

In [ ]:
def dedup_citations(citations):
    seen = set()
    unique = []

    for c in citations:
        key = (c["chunk_id"])

        if key not in seen:
            seen.add(key)
            unique.append(c)

    return unique

In [ ]:
def retrieve(state):
    original_q = state["question"]
    # rewritten_q = rewrite_query(original_q)

    # extracting section and subsection from query if given
    section, subsection = extract_query_filters(original_q)


    # retriveving docs using similarity search and bm23(hybrid model)
    vector_docs = vector_store.similarity_search(original_q, k=20)
    bm25_docs = bm25.invoke(original_q)


    # stiching them in one
    combined = vector_docs + bm25_docs

    # ommit the duplicate docs
    combined = deduplicate_docs(combined)


    # filtering the docs based on section and subsection
    filtered = filter_documents(combined, section, subsection)


    # reranking the filtered docs
    reranked = rerank_documents(original_q, filtered)

    
    print("\n RERANKING RESULTS:\n")

    final_docs = []

    for i, (doc, score) in enumerate(reranked):
        print(f"Rank {i+1} | Score: {score:.4f}")
        print(doc.page_content[:200])
        print("-" * 50)

        final_docs.append(doc)


    # citations
    citations = extract_citations(final_docs)
    citations = dedup_citations(citations)

    return {
        "docs": final_docs,
        "citations":citations
    }

In [ ]:
def format_output(answer, citations):
    formatted = answer + "\n\nSources:\n"

    for i, c in enumerate(citations, 1):
        formatted += f"[{i}] Section {c['section']}.{c['subsection']} (Page {c['page']})\n"

    return formatted

In [ ]:

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": format_output(out.content,state["citations"])}


In [ ]:
from langgraph.graph import StateGraph,START,END

In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
# 5) Run
res = app.invoke({"question": "explain me ATTENDANCE AND TIME MANAGEMENT policy in section 5"})

In [ ]:
res["question"]

In [ ]:
print(res["answer"])

# without specifing  section filtering  

In [ ]:
#print(res["answer"])

In [ ]:
# print(res['docs'][0].page_content)
# print('*'*100)
# print(res['docs'][1].page_content)
# print('*'*100)
# print(res['docs'][2].page_content)
# print('*'*100)
# print(res['docs'][3].page_content)